# ASYA — универсальный генератор сцен
1. Выбери T4 GPU (если бесплатно доступен). 2. В ячейке генерации поменяй `PROJECT` на `crow`, `wolverine` или `wolf`. 3. Выполни все ячейки. 4. По желанию запусти ячейку отправки в GitHub. Для новой темы добавь её в `projects.json` репозитория и запусти ноутбук заново. Каждый проект хранится отдельно.
SD-Turbo генерирует 512×768, увеличение до 1080×1920 не добавляет детализации. Генерация на бесплатном GPU не гарантируется.


In [ ]:
%pip -q install 'diffusers>=0.30,<0.38' 'transformers>=4.44,<5' accelerate safetensors


In [ ]:
import requests
url='https://raw.githubusercontent.com/todimka93-hub/honeybadger-studio/main/projects.json'
response=requests.get(url,timeout=30); response.raise_for_status()
open('/content/projects.json','w',encoding='utf-8').write(response.text)


In [ ]:
import json, re, zipfile, torch
from pathlib import Path
from PIL import Image
from diffusers import AutoPipelineForText2Image
from google.colab import files
from google.colab import userdata

PROJECT = "crow"  # crow / wolverine / wolf
DATA = json.loads(Path("/content/projects.json").read_text(encoding="utf-8"))
if PROJECT not in DATA:
    raise ValueError(f"Unknown project: {PROJECT}. Available: {list(DATA)}")
if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable. Select Runtime > Change runtime type > T4 GPU, if offered.")
project = DATA[PROJECT]
prompts = project["scenes"]
output = Path("/content/asya") / PROJECT
output.mkdir(parents=True, exist_ok=True)
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")
for i, scene in enumerate(prompts, 1):
    gen = torch.Generator(device="cuda").manual_seed(20261022 + i * 97)
    image = pipe(prompt=project["style"] + " " + scene, num_inference_steps=4, guidance_scale=0, width=512, height=768, generator=gen).images[0]
    image.resize((1080, 1920), Image.Resampling.LANCZOS).save(output / f"scene_{i:02d}.png")
    print(f"{PROJECT}: {i}/{len(prompts)}")
archive = f"/content/{PROJECT}_scenes.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(output.glob("*.png")): z.write(p, p.name)
files.download(archive)



## Необязательно: отправить PNG прямо в GitHub
Создай fine-grained token с разрешениями **Contents: Read and write** и **Actions: Read and write** только для `honeybadger-studio`. Ввод скрытый, токен не записывается в ноутбук. Если озвучка проекта уже загружена, стартует монтаж.


In [ ]:
import base64, getpass, requests
token = getpass.getpass("Fine-grained GitHub token (Contents + Actions read/write): ").strip()
if not token: raise ValueError("Token required")
repo = "todimka93-hub/honeybadger-studio"
session = requests.Session()
session.headers.update({"Authorization":f"Bearer {token}","Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"})
try:
    for p in sorted(output.glob("scene_*.png")):
        path = f"projects/{PROJECT}/assets/{p.name}"
        url = f"https://api.github.com/repos/{repo}/contents/{path}"
        existing = session.get(url,timeout=30)
        if existing.status_code not in (200,404): raise RuntimeError(existing.text[:250])
        body = {"message":f"Add {PROJECT} scene {p.name}","content":base64.b64encode(p.read_bytes()).decode()}
        if existing.status_code == 200: body["sha"]=existing.json()["sha"]
        res=session.put(url,json=body,timeout=120)
        if res.status_code not in (200,201): raise RuntimeError(f"{p.name}: {res.status_code} {res.text[:250]}")
        print("Uploaded",path)
    voice=session.get(f"https://api.github.com/repos/{repo}/contents/projects/{PROJECT}/assets/voice.mp3",timeout=30)
    if voice.status_code == 200:
        r=session.post(f"https://api.github.com/repos/{repo}/actions/workflows/projects.yml/dispatches",json={"ref":"main","inputs":{"project":PROJECT}},timeout=30)
        print("Render dispatch:",r.status_code, "(204 means started)")
    else:
        print(f"Add projects/{PROJECT}/assets/voice.mp3, then run Build Project with project={PROJECT}")
finally:
    session.headers.pop("Authorization",None)
    token=None

